CONFIRMAR O AMBIENTE

In [0]:
# Confirmar que o Spart está ativo
print(f"Spark version: {spark.version}")

Spark version: 4.1.0


In [0]:
# Confirmar que o arquivo está acessível
display(dbutils.fs.ls("/Volumes/workspace/lakehouse_edu/bronze/"))


Ler o CSV bruto

In [0]:
# Definir o caminho do arquivo
RAW_PATH = "/Volumes/workspace/lakehouse_edu/bronze/student-mat.csv"

In [0]:
#Ler o CSV
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true") #primeira linha é o cabeçalho
    .option("inferSchema", "true")#Spark detecta os tipos de variáveis automaticamente
    .option("sep", ",") #separador é vírgula
    .load(RAW_PATH)
)

In [0]:
display(df_raw.limit(10))

Inspecionar o schema e contar registros

In [0]:
#QUantidade de linhas e colunas
print(f"Total de linhas: {df_raw.count()}")
print(f"Total de colunas: {len(df_raw.columns)}")
#Verificar os tipos de variáveis inferidos pelo Spark
df_raw.printSchema()


Adicionar metadados de auditoria

In [0]:
from pyspark.sql.functions import current_timestamp, lit
#Adicionar colunas de controle
df_bronze = (
    df_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("student-mat-csv"))
    .withColumn("_layer", lit("bronze"))
)

In [0]:
#Confirmar as novas colunas
print("Colunas após adicionar metadados: ")
print(df_bronze.columns)

Salvar como Delta Table na camada Bronze

In [0]:
#Definir o caminho de destino dentro do volume

BRONZE_PATH = "/Volumes/workspace/lakehouse_edu/bronze/student_performance_delta"

In [0]:
# Salvar como Delta Table
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .save(BRONZE_PATH)
)

In [0]:
print("✅ Camada Bronze salva com sucesso!")
print(f"📁 Localização: {BRONZE_PATH}")

✅ Camada Bronze salva com sucesso!
📁 Localização: /Volumes/workspace/lakehouse_edu/bronze/student_performance_delta


Registrar a tabela no catálogo


In [0]:
#Salver e registrar diretamente como tabela no catálogo
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.lakehouse_edu.bronze_student_performance")
)

print("✅ Tabela registrada no catálogo com sucesso!")


✅ Tabela registrada no catálogo com sucesso!


Validação final da camada Bronze

In [0]:
#Ler direto da tabela registrada no catálogo
df_check = spark.table("workspace.lakehouse_edu.bronze_student_performance")


In [0]:
# Validar contagem
print(f"Total de registros na Bronze: {df_check.count()}")

Total de registros na Bronze: 395


In [0]:
#Confirmar que os metadados estão presentes
print(f"Colunas de metadados:")
print([c for c in df_check.columns if c.startswith("_")])

Colunas de metadados:
['_ingestion_timestamp', '_source_file', '_layer']


In [0]:
#Visualizar amostra final
display(df_check.limit(5))